# 03 Retrieval of guidance passages

Member 5. After the checker, the advisory needs practical guidance to build on. Retrieval finds it: the supported claims are turned into a query, the query and every guidance passage are turned into vectors by an embedding model (`BAAI/bge-m3`, English and Arabic), and the passages closest to the query are returned.

1. Build passages from guidance documents you collect (a folder on Drive plus a `sources.csv`).
2. Embed them once and save the vectors.
3. Turn verified claims into a query and search.
4. Make a small annotation sheet, score hit@3 and precision@3.

**Which documents is a team decision.** The repo does not name a collection. This notebook reads whatever is in the folder. Keep the source files on Drive, never in git, and fill in the licence note for every source. For Arabic, use published Arabic text, not machine translation.

A GPU speeds up the embedding step but a few hundred passages also run on CPU. Clear outputs before committing.

In [ ]:
BRANCH = "main"

import os, subprocess, sys

REPO_DIR = "/content/FloodVerify"
if "google.colab" in sys.modules:
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "https://github.com/aaarrvind/FloodVerify.git", REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers", "pypdf", "pandas", "matplotlib"], check=True)
    REPO = REPO_DIR
else:
    REPO = os.path.abspath("../..")
sys.path.insert(0, os.path.join(REPO, "src"))
print("repo:", REPO)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
GUIDANCE_DIR = f"{PROJECT}/guidance"                         # your own folder: the documents and sources.csv go here
SOURCES_CSV = f"{GUIDANCE_DIR}/sources.csv"
PASSAGES_CSV = f"{GUIDANCE_DIR}/passages.csv"
EMB_FILE = f"{GUIDANCE_DIR}/passages_bge_m3.npy"
EVIDENCE_DIR = f"{PROJECT}/xbd/processed/evidence_labels"    # label evidence; switch to the predicted folder to test Gate 2
OUT = f"{GUIDANCE_DIR}/retrieval_eval"
os.makedirs(OUT, exist_ok=True)

EMBED_MODEL = "BAAI/bge-m3"
TOP_K = 3
MAX_CHARS = 600           # target passage size
MIN_CHARS = 40            # shorter chunks are dropped
MERGE_BELOW = 120         # a paragraph shorter than this is joined to the next one
SEED = 42
print("guidance folder exists:", os.path.isdir(GUIDANCE_DIR))

## 1. Build passages

Put each document in the guidance folder (`.txt`, `.md` or `.pdf`) and list it in `sources.csv` with these columns:

| column | meaning |
| --- | --- |
| `file` | file name inside the guidance folder |
| `source_name` | who published it, for example the issuing authority |
| `url` | where it came from |
| `language` | `en` or `ar`; leave empty to detect it |
| `licence_note` | the terms you found, or `to check` |

Arabic PDFs often extract with broken word order. Look at a few Arabic passages below. If they are garbled, copy the text from the web page into a `.txt` file instead.

In [ ]:
import re, json, hashlib
import numpy as np
import pandas as pd

ARABIC = re.compile(r"[\u0600-\u06FF]")
SENT_SPLIT = re.compile(r"(?<=[.!?\u061F])\s+")


def language_of(text):
    """'ar' when more than 30% of the letters are Arabic, else 'en'."""
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return "en"
    return "ar" if sum(bool(ARABIC.match(c)) for c in letters) / len(letters) > 0.3 else "en"


def read_document(path):
    """Plain text of a .txt, .md or .pdf file."""
    low = path.lower()
    if low.endswith(".pdf"):
        from pypdf import PdfReader
        return "\n\n".join((p.extract_text() or "") for p in PdfReader(path).pages)
    return open(path, encoding="utf-8").read()


def chunk_text(text, max_chars=MAX_CHARS, min_chars=MIN_CHARS, merge_below=MERGE_BELOW):
    """One passage per paragraph or bullet; long ones are cut at sentence ends, very short ones joined to the next."""
    pieces = []
    for block in re.split(r"\n\s*\n", text):
        for item in re.split(r"\n\s*(?=[-\u2022*]\s|\d+[.)]\s)", block):
            para = re.sub(r"\s+", " ", item.strip().lstrip("-\u2022*").strip())
            if not para:
                continue
            if len(para) <= max_chars:
                pieces.append(para)
                continue
            cur = ""
            for s in SENT_SPLIT.split(para):
                if cur and len(cur) + len(s) + 1 > max_chars:
                    pieces.append(cur)
                    cur = s
                else:
                    cur = f"{cur} {s}".strip()
            if cur:
                pieces.append(cur)
    out, cur = [], ""
    for p in pieces:
        if cur and len(cur) < merge_below and len(cur) + len(p) + 1 <= max_chars:
            cur = f"{cur} {p}"
        else:
            if cur:
                out.append(cur)
            cur = p
    if cur:
        out.append(cur)
    return [c for c in out if len(c) >= min_chars]


def build_passages(sources_csv, folder):
    """One row per passage: id, source_name, url, language, licence_note, file, text."""
    src = pd.read_csv(sources_csv, dtype=str).fillna("")
    rows = []
    for _, r in src.iterrows():
        text = read_document(os.path.join(folder, r["file"]))
        for i, chunk in enumerate(chunk_text(text)):
            rows.append({"id": f"{os.path.splitext(r['file'])[0]}_{i:03d}", "source_name": r["source_name"],
                         "url": r["url"], "language": r["language"] or language_of(chunk),
                         "licence_note": r["licence_note"] or "to check", "file": r["file"], "text": chunk})
    return pd.DataFrame(rows)


passages = build_passages(SOURCES_CSV, GUIDANCE_DIR)
passages.to_csv(PASSAGES_CSV, index=False)
print(len(passages), "passages")
print(passages.groupby(["source_name", "language"]).size())

In [ ]:
def passage_stats(passages):
    """Length statistics per language (characters)."""
    return passages.assign(chars=passages["text"].str.len()).groupby("language")["chars"].describe().round(0)


def plot_passage_lengths(passages):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6, 3))
    for lang, g in passages.groupby("language"):
        ax.hist(g["text"].str.len(), bins=20, alpha=0.6, label=lang)
    ax.set_xlabel("characters per passage"); ax.set_ylabel("passages"); ax.legend()
    plt.tight_layout(); plt.show()


print(passage_stats(passages))
plot_passage_lengths(passages)
for _, r in passages.groupby("language").head(2).iterrows():
    print(f"[{r['language']}] {r['id']}: {r['text'][:200]}")

## 2. Embed the passages

`bge-m3` maps text in either language to the same vector space, so an English query can find an Arabic passage. Vectors are normalised, so a dot product is the cosine similarity. They are saved with the passage ids, so the step is skipped on later runs unless the passages change.

In [ ]:
def load_embedder(name=EMBED_MODEL):
    """Sentence-transformers model on the GPU when there is one."""
    from sentence_transformers import SentenceTransformer
    import torch
    return SentenceTransformer(name, device="cuda" if torch.cuda.is_available() else "cpu")


def passages_key(passages, model_name):
    """Fingerprint of the passages and the model, to know when saved vectors are stale."""
    h = hashlib.sha1((model_name + "|" + "|".join(passages["id"] + passages["text"])).encode("utf-8")).hexdigest()
    return h[:16]


def embed_passages(model, passages, batch_size=16):
    """Normalised vectors, one row per passage."""
    return np.asarray(model.encode(list(passages["text"]), batch_size=batch_size,
                                   normalize_embeddings=True, show_progress_bar=False), dtype="float32")


def load_or_embed(model, passages, path, model_name=EMBED_MODEL):
    """Reuse saved vectors when the key matches, else embed and save."""
    key = passages_key(passages, model_name)
    if os.path.exists(path) and os.path.exists(path + ".key") and open(path + ".key").read() == key:
        return np.load(path)
    emb = embed_passages(model, passages)
    np.save(path, emb)
    open(path + ".key", "w").write(key)
    return emb


embedder = load_embedder()
emb = load_or_embed(embedder, passages, EMB_FILE)
print(emb.shape)

## 3. Queries from verified claims

The advisory will receive the supported claims for a tile. The function `claims_to_query` turns them into one English sentence. Two query sources are used here:

- **Tile queries.** True claims derived from label evidence (what the checker would confirm for a perfect report). In the live pipeline the input is the supported part of `verify()` output instead.
- **Topic queries.** A short hand-written list of what a reader might need. Many tile queries are alike ("few buildings with minor damage"), so these add variety to the evaluation. Edit the list to fit the guidance collection.

In [ ]:
from floodsitrep.nlp.checker import bin_of, class_fraction, MIN_VALID


def evidence_to_claims(evidence):
    """True claims for one evidence file: for each usable cell and damage level, the bin of the share (none is skipped)."""
    out = []
    for c in evidence["cells"]:
        if c["buildings"] == 0 or c.get("valid_fraction", 1.0) < MIN_VALID:
            continue
        for level in ("minor", "major", "destroyed"):
            word = bin_of(class_fraction([c], level) or 0.0)
            if word != "none":
                out.append({"cell": c["cell"], "quantity": word, "damage_class": level})
    return out


def claims_to_query(claims, max_parts=4):
    """Claims (dicts with quantity and damage_class) -> one English query sentence."""
    seen, parts = set(), []
    for c in claims:
        key = (c["quantity"], c["damage_class"])
        if key in seen:
            continue
        seen.add(key)
        parts.append(f'{c["quantity"]} destroyed buildings' if c["damage_class"] == "destroyed"
                     else f'{c["quantity"]} buildings with {c["damage_class"]} damage')
    if not parts:
        return "Safety advice for residents after a flood: buildings without visible damage"
    return "Safety advice after flood damage to buildings: " + "; ".join(parts[:max_parts])


TOPIC_QUERIES = [
    "What to do if a road is flooded while driving",
    "Entering a building that was damaged by floodwater",
    "Electrical hazards after a flood",
    "Is tap water safe to drink after flooding",
    "When and how to evacuate during a flash flood",
    "Returning home after a flood and cleaning up",
    "Protecting children and elderly people during floods",
    "Staying away from wadis and flood channels in heavy rain",
    "Reporting damage and contacting emergency services",
    "Mould and structural checks in flooded buildings",
]


def tile_queries(evidence_dir, n=10, seed=SEED):
    """n distinct tile queries from tiles that have at least one claim."""
    rng = np.random.default_rng(seed)
    names = sorted(f[:-5] for f in os.listdir(evidence_dir) if f.endswith(".json"))
    rng.shuffle(names)
    out, seen = [], set()
    for t in names:
        claims = evidence_to_claims(json.load(open(f"{evidence_dir}/{t}.json")))
        q = claims_to_query(claims)
        if claims and q not in seen:
            seen.add(q)
            out.append({"query_id": f"tile_{len(out):02d}", "kind": "tile", "tile": t, "query": q})
        if len(out) == n:
            break
    return out


queries = tile_queries(EVIDENCE_DIR) + [{"query_id": f"topic_{i:02d}", "kind": "topic", "tile": "", "query": q}
                                       for i, q in enumerate(TOPIC_QUERIES)]
pd.DataFrame(queries)[["query_id", "kind", "query"]]

## 4. Search

`search` returns the top passages for a query. `language` restricts the results to one language: the advisory in Arabic should show Arabic passages, found by an English query. `retrieve` is the function shape for the advisory step (`retrieve(claims, lang, k)`); move it to `src/floodsitrep/advisory/retrieve.py` once it is agreed with Member 1, because it is not part of the frozen contract.

In [ ]:
def search(model, emb, passages, query, k=TOP_K, language=None):
    """Top-k passages by cosine similarity, optionally only one language."""
    q = np.asarray(model.encode([query], normalize_embeddings=True, show_progress_bar=False), dtype="float32")[0]
    sims = emb @ q
    order = np.argsort(-sims)
    if language:
        order = [i for i in order if passages.iloc[i]["language"] == language]
    top = list(order[:k])
    out = passages.iloc[top].copy()
    out["score"] = [round(float(x), 3) for x in sims[top]]
    out["rank"] = range(1, len(top) + 1)
    return out


def retrieve(claims, lang="en", k=TOP_K, model=None, emb_=None, passages_=None):
    """Advisory-facing call: supported claims -> list of passage dicts (text, source, url, score)."""
    model, emb_, passages_ = model or embedder, emb if emb_ is None else emb_, passages if passages_ is None else passages_
    hits = search(model, emb_, passages_, claims_to_query(claims), k=k, language=lang)
    return hits[["id", "text", "source_name", "url", "score"]].to_dict("records")


def run_queries(model, emb, passages, queries, k=TOP_K, language=None):
    """All queries at once: one row per query and rank."""
    rows = []
    for q in queries:
        for _, r in search(model, emb, passages, q["query"], k, language).iterrows():
            rows.append({**q, "rank": int(r["rank"]), "passage_id": r["id"], "score": r["score"],
                         "language": r["language"], "source_name": r["source_name"], "text": r["text"]})
    return pd.DataFrame(rows)


results = run_queries(embedder, emb, passages, queries)
shown = results[results["query_id"].isin(["tile_00", "topic_00"])]
for _, r in shown.iterrows():
    print(f"{r['query_id']} #{r['rank']} ({r['score']}, {r['language']}, {r['source_name']}): {r['text'][:160]}")

In [ ]:
def language_mix(results):
    """Share of top-k results by language, and mean top-1 score, per query kind."""
    mix = results.groupby(["kind", "language"]).size().unstack(fill_value=0)
    top1 = results[results["rank"] == 1].groupby("kind")["score"].mean().round(3).rename("mean_top1_score")
    return mix.join(top1)


language_mix(results)

### English queries, Arabic passages

The same queries with the results restricted to Arabic. This is what the Arabic advisory would receive. If there are no Arabic passages in the collection, the table is empty and the advisory has nothing to build on in Arabic.

In [ ]:
ar_results = run_queries(embedder, emb, passages, queries[:3], language="ar") if (passages["language"] == "ar").any() else pd.DataFrame()
print("Arabic passages:", int((passages["language"] == "ar").sum()))
if len(ar_results):
    for _, r in ar_results.iterrows():
        print(f"{r['query_id']} #{r['rank']} ({r['score']}): {r['text'][:140]}")

## 5. Annotation sheet and scores

Retrieval is checked by people: for every query and each of the top 3 passages, mark whether it helps answer the query. Labels: `relevant`, `partly`, `not relevant`. One rater gives no agreement figure, so say so, or ask a second person to label the same sheet.

- **hit@3:** share of queries with at least one `relevant` passage in the top 3.
- **precision@3:** mean over queries of (relevant + 0.5 partly) / 3.

In [ ]:
LABELS = ["relevant", "partly", "not relevant"]


def make_eval_sheet(results, path):
    """Sheet for the rater: query, rank, passage text and an empty label column."""
    sheet = results[["query_id", "query", "rank", "passage_id", "language", "text"]].copy()
    sheet["label"] = ""
    sheet.to_csv(path, index=False)
    return sheet


def score_eval(sheet, k=TOP_K):
    """hit@k and graded precision@k per query kind, from a labelled sheet."""
    s = sheet.copy()
    s["gain"] = s["label"].map({"relevant": 1.0, "partly": 0.5, "not relevant": 0.0})
    if s["gain"].isna().any():
        raise ValueError(f"{int(s['gain'].isna().sum())} rows have no valid label (use {LABELS})")
    s["kind"] = s["query_id"].str.split("_").str[0]
    rows = []
    for (kind, qid), g in s.groupby(["kind", "query_id"]):
        rows.append({"kind": kind, "query_id": qid, "hit": float((g["label"] == "relevant").any()),
                     "precision": g["gain"].sum() / k})
    per_q = pd.DataFrame(rows)
    out = per_q.groupby("kind")[["hit", "precision"]].mean().round(3)
    out["queries"] = per_q.groupby("kind").size()
    return out.rename(columns={"hit": f"hit@{k}", "precision": f"precision@{k}"})


sheet = make_eval_sheet(results, f"{OUT}/eval_sheet.csv")
print(len(sheet), "rows to label ->", f"{OUT}/eval_sheet.csv")

Label the `label` column in the CSV (open it in Sheets or Excel), save it as `eval_sheet_labelled.csv` in the same folder, and run the next cell.

In [ ]:
LABELLED = f"{OUT}/eval_sheet_labelled.csv"
if os.path.exists(LABELLED):
    labelled = pd.read_csv(LABELLED, dtype=str).fillna("")
    labelled["rank"] = labelled["rank"].astype(int)
    scores = score_eval(labelled)
    scores.to_csv(f"{OUT}/retrieval_scores.csv")
    print(scores)
else:
    print("Label the sheet first:", LABELLED)

## How to read this

- **hit@3 and precision@3** say whether the top passages help. Report both per query kind, with the number of queries (about 20) and the number of raters. With so few queries, differences of a few points are not meaningful.
- **Tile queries are similar to each other** because the evidence has few distinct damage patterns, so topic queries carry most of the test. Say so.
- **Cross-language search** is the reason for bge-m3. The Arabic table shows whether an English query reaches Arabic guidance. Check a few Arabic passages by eye or ask an Arabic reader.
- **Wrong or unsafe passages reach the advisory.** The advisory should use only retrieved passages for practical advice and say nothing the passages do not say.
- **Not in the pipeline yet.** `retrieve` is a notebook function. Agree the signature with Member 1, then move it into `src/floodsitrep/advisory/retrieve.py` with a small test.
- **Licences.** Each passage carries its source and licence note. Keep the documents and the vectors out of git.